#VGG16

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


In [3]:
class VGG16(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.block1=nn.Sequential(
            nn.Conv2d(3,64,kernel_size=3,padding=1), #3 input and 64 output
            nn.ReLU(),
            
            nn.Conv2d(64,64,kernel_size=3,padding=1),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        
        self.block2=nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3,padding=1), #3 input and 128 output
            nn.ReLU(),
            
            nn.Conv2d(128,128,kernel_size=3,padding=1),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        
        self.block3=nn.Sequential(
            nn.Conv2d(128,256,kernel_size=3,padding=1), #3 input and 128 output
            nn.ReLU(),
            
            nn.Conv2d(256,256,kernel_size=3,padding=1),
            nn.ReLU(),
            
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2)
        )

        self.block4=nn.Sequential(
            nn.Conv2d(256,512,kernel_size=3,padding=1), #3 input and 512 output
            nn.ReLU(),
            
            nn.Conv2d(512,512,kernel_size=3,padding=1),
            nn.ReLU(),
            
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        
        self.block5=nn.Sequential(
            nn.Conv2d(512,512,kernel_size=3,padding=1), #3 input and 512 output
            nn.ReLU(),
            
            nn.Conv2d(512,512,kernel_size=3,padding=1),
            nn.ReLU(),
            
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        
        self.fc=nn.Sequential(
            nn.Flatten(),
            nn.Linear(7*7*512,4096),
            nn.ReLU(),
            
            nn.Linear(4096,4096),
            nn.ReLU(),
            
            # nn.Linear(4096,1000),
            nn.Linear(4096, 2)
        )
        
    def forward(self,x):
        x=self.block1(x)
        x=self.block2(x)
        x=self.block3(x)
        x=self.block4(x)
        x=self.block5(x)
        x=self.fc(x)
        
        return x

Model Testing

In [4]:
model = VGG16()
x = torch.randn(1, 3, 224, 224)

output = model(x)
print(output.shape)

torch.Size([1, 2])


In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

Image transformation

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
#Loading dataset
dataset=datasets.ImageFolder(root="../datasets/PetImages",transform=transform)

In [8]:
print("Classes:", dataset.classes)
print("Class mapping:", dataset.class_to_idx)
print("Number of images:", len(dataset))

Classes: ['Cat', 'Dog']
Class mapping: {'Cat': 0, 'Dog': 1}
Number of images: 24998


In [19]:
image, label = dataset[0]

print(image.shape)
print(label)

torch.Size([3, 224, 224])
0


Split the dataset

In [22]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [32]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([16, 3, 224, 224])
Labels: torch.Size([16])


In [33]:
images, labels = next(iter(val_loader))

print("Testing images:", images.shape)
print("Testing labels:", labels.shape)

Testing images: torch.Size([16, 3, 224, 224])
Testing labels: torch.Size([16])


Loss Function

In [34]:
loss_fxn = nn.CrossEntropyLoss()

Adam Optimizer

In [35]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

Training for ONE batch first

In [ ]:
model.train()

images,labels=next(iter(train_loader))

images=images.to(device)
labels=labels.to(device)

Forward propagation

In [37]:
output = model(images)
print(output.shape)

torch.Size([16, 2])


Calculate loss

In [38]:
loss=loss_fxn(output,labels)
print("Loss:",loss.item())

Loss: 0.693986177444458


Backpropagation

In [39]:
optimizer.zero_grad()
output = model(images)
loss = loss_fxn(output, labels)
loss.backward()
optimizer.step()

In [40]:
num_epochs = 5

loss_history = []
accuracy_history = []

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # 1. Forward propagation
        output = model(images)

        # 2. Calculate loss
        loss = loss_fxn(output, labels)

        # 3. Clear previous gradients
        optimizer.zero_grad()

        # 4. Backpropagation
        loss.backward()

        # 5. Update weights
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

        # Prediction
        predictions = torch.argmax(output, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    loss_history.append(epoch_loss)
    accuracy_history.append(epoch_accuracy)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

KeyboardInterrupt: 

Validation

In [ ]:
model.eval()

val_loss=0
val_correct=0
val_total=0

with torch.no_grad():
    for images,labels in test_loader:
        images=images.to(device)
        labels = labels.to(device)
        
        output=model(images)
        
        loss=loss_fxn(output,labels)
        val_loss+=loss.item()
        prediction=torch.argmax(output,dim=1)
        
        val_correct+=(predictions==labels).sum().item()
        val_total+=labels.size(0)
        
val_loss = val_loss / len(val_loader)
val_accuracy = 100 * val_correct / val_total

print(
    f"Validation Loss: {val_loss:.4f} "
    f"Validation Accuracy: {val_accuracy:.2f}%"
)

Plot Training Results

In [ ]:
plt.plot(loss_history)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")

plt.show()

In [ ]:
plt.plot(accuracy_history)

plt.xlabel("Epoch")
plt.ylabel("Training Accuracy")
plt.title("Training Accuracy")

plt.show()

Save the trained model

In [ ]:
torch.save(
    model.state_dict(),
    "VGG16_Cats_Dogs.pth"
)

Load the trained model

In [ ]:
model=VGG16().to(device)
model.load_state_dict(
    torch.load(
        "VGG16_Cats_Dogs.pth",
        map_location=device
    )
)
model.eval()

Image Prediction

In [ ]:
image = Image.open("../datasets/PetImages/Cat/1.jpg").convert("RGB")
image_tensor=transform(image)
image_tensor = image_tensor.unsqueeze(0)
image_tensor = image_tensor.to(device)

model.eval()
with torch.no_grad():
    output = model(image_tensor)
    prediction = torch.argmax(output, dim=1)

predicted_class = dataset.classes[prediction.item()]
print("Prediction:", predicted_class)